In [96]:
# Common imports
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
import bootcampviztools as bct
import itertools
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from sklearn.cluster import AgglomerativeClustering
from scipy.cluster.hierarchy import linkage, dendrogram
import matplotlib.pyplot as plt

# to make this notebook's output stable across runs
np.random.seed(42)

# To plot pretty figures
%matplotlib inline
import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc('axes', labelsize=14)
mpl.rc('xtick', labelsize=12)
mpl.rc('ytick', labelsize=12)

# Ignore useless warnings (see SciPy issue #5998)
import warnings
warnings.filterwarnings(action="ignore", message="^internal gelsd")

In [97]:
#recordad hacer pip install fastparquet y usar este codigo cuando lo abrais:
df_reg = pd.read_parquet('./src/data_sample/datos_licitaciones.parquet', engine='fastparquet')



In [98]:
df_reg.head()

,Situació contractual,Exercici,Àmbit organitzatiu,Identificador agrupació organisme,Agrupació organisme,Identificador organisme contractant,Organisme contractant,Codi de l’expedient,Procediment d’adjudicació,Tipus de contracte,...,Tipus de modificació,Import de la modificació,Data aprovació modificació,Termini modificació anys,Termini modificació mesos,Termini modificació dies,Tipus de liquidació,Data de liquidació,Causa de resolució,Import de la liquidació
442377,liquidació,2021,Universitats,8000,UNIVERSITATS,0000000013,Universitat Pompeu Fabra,S-508124,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,18/02/2021,,7500.00
1555898,liquidació,2022,Universitats,8000,UNIVERSITATS,0000000015,Universitat de Girona,C191921,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,24/02/2022,,252.00
2587403,liquidació,2023,Entitats de l'Administració Local,0823100000,Ajuntament de Sant Pere de Ribes,0823100000,Ajuntament de Sant Pere de Ribes,12023000001965/AD/1,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,14/03/2023,,699.89
1948781,liquidació,2024,Entitats de l'Administració Local,9912948005,"Creacció Agència d'Emprenedoria, Innovació i C...",9912948005,"Creacció Agència d'Emprenedoria, Innovació i C...",A24-66,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,15/02/2024,,37.50
3016399,liquidació,2021,Departaments i Sector Públic de la Generalitat...,1500,DEPARTAMENT DE SALUT,7996100024,Fundació IDIAP Jordi Gol i Gurina,FRC21/0175,Menor,5. SERVEIS,...,,NaN,,NaN,NaN,NaN,COMPLIMENT,13/08/2021,,2993.22


In [99]:
# Encontrar columnas totalmente nulas o con todos los strings vacíos
columnas_nulas = [col for col in df_reg.columns 
                         if df_reg[col].isnull().all() or (df_reg[col].astype(str) == '').all()]

print(f"Total de columnas vacías: {len(columnas_nulas)}\n")
for col in columnas_nulas:
    print(col)

Total de columnas vacías: 10

Número de pròrroga
Data inici pròrroga
Data fi pròrroga
Número de modificació
Tipus de modificació
Import de la modificació
Data aprovació modificació
Termini modificació anys
Termini modificació mesos
Termini modificació dies


In [100]:
# Elimina las columnas nulas
df_reg = df_reg.drop(columns=columnas_nulas)

In [101]:
df_reg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70000 entries, 442377 to 239772
Data columns (total 25 columns):
 #   Column                               Non-Null Count  Dtype  
---  ------                               --------------  -----  
 0   Situació contractual                 70000 non-null  object 
 1   Exercici                             70000 non-null  int64  
 2   Àmbit organitzatiu                   70000 non-null  object 
 3   Identificador agrupació organisme    70000 non-null  object 
 4   Agrupació organisme                  70000 non-null  object 
 5   Identificador organisme contractant  70000 non-null  object 
 6   Organisme contractant                70000 non-null  object 
 7   Codi de l’expedient                  70000 non-null  object 
 8   Procediment d’adjudicació            70000 non-null  object 
 9   Tipus de contracte                   70000 non-null  object 
 10  Descripció de l’expedient            70000 non-null  object 
 11  Número de lot              

In [102]:
# Value counts de todas las columnas no numéricas
columnas_no_numericas = df_reg.select_dtypes(exclude=['number']).columns

for col in columnas_no_numericas:
    print(f"\n{'='*60}")
    print(f"Columna: {col}")
    print(f"{'='*60}")
    print(df_reg[col].value_counts())
    print(f"\nTotal valores únicos: {df_reg[col].nunique()}")


Columna: Situació contractual
Situació contractual
liquidació    70000
Name: count, dtype: int64

Total valores únicos: 1

Columna: Àmbit organitzatiu
Àmbit organitzatiu
Entitats de l'Administració Local                              39520
Departaments i Sector Públic de la Generalitat de Catalunya    22904
Universitats                                                    7576
Name: count, dtype: int64

Total valores únicos: 3

Columna: Identificador agrupació organisme
Identificador agrupació organisme
1500          9568
8000          7571
9612260004    2953
1400          2515
8000840003    2156
              ... 
1712150006       1
4304580001       1
0809960009       1
8103110007       1
1703530008       1
Name: count, Length: 640, dtype: int64

Total valores únicos: 640

Columna: Agrupació organisme
Agrupació organisme
DEPARTAMENT DE SALUT                     9568
UNIVERSITATS                             7571
DEPARTAMENT DE RECERCA I UNIVERSITATS    2953
DEPARTAMENT DE CULTURA        

In [103]:
columnas_sin_aporte = ['Situació contractual',
'Identificador agrupació organisme',
'Identificador organisme contractant',
'Codi de l’expedient',
'Descripció de l’expedient',
'Codi CPV',
'Descripció del lot',
'Lot desert',
'Causa de resolució']

df_reg = df_reg.drop(columns=columnas_sin_aporte)

In [104]:
df_reg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70000 entries, 442377 to 239772
Data columns (total 16 columns):
 #   Column                     Non-Null Count  Dtype  
---  ------                     --------------  -----  
 0   Exercici                   70000 non-null  int64  
 1   Àmbit organitzatiu         70000 non-null  object 
 2   Agrupació organisme        70000 non-null  object 
 3   Organisme contractant      70000 non-null  object 
 4   Procediment d’adjudicació  70000 non-null  object 
 5   Tipus de contracte         70000 non-null  object 
 6   Número de lot              70000 non-null  int64  
 7   Adjudicatari               70000 non-null  object 
 8   Import d’adjudicació       70000 non-null  float64
 9   Data d’adjudicació         70000 non-null  object 
 10  Durada dies                70000 non-null  int64  
 11  Durada mesos               70000 non-null  int64  
 12  Durada anys                70000 non-null  int64  
 13  Tipus de liquidació        70000 non-null  ob

In [105]:
# Convertir columnas de fecha DD/MM/YYYY a formato numérico (timestamp)
columnas_data = [col for col in df_reg.columns if col.startswith('Data')]

for col in columnas_data:
    # Primero convertir a datetime
    df_reg[col] = pd.to_datetime(df_reg[col], format='%d/%m/%Y', errors='coerce')
    # Luego convertir a timestamp numérico (segundos desde 1970-01-01)
    df_reg[col + '_numeric'] = df_reg[col].astype('int64') / 10**9
    df_reg = df_reg.drop(columns=col)
    
print(f"Columnas de fecha convertidas a numérico: {columnas_data}")
print(f"\nSe crearon nuevas columnas con sufijo '_numeric'")

Columnas de fecha convertidas a numérico: ['Data d’adjudicació', 'Data de liquidació']

Se crearon nuevas columnas con sufijo '_numeric'


In [106]:
df_reg.info()

<class 'pandas.core.frame.DataFrame'>
Index: 70000 entries, 442377 to 239772
Data columns (total 16 columns):
 #   Column                      Non-Null Count  Dtype  
---  ------                      --------------  -----  
 0   Exercici                    70000 non-null  int64  
 1   Àmbit organitzatiu          70000 non-null  object 
 2   Agrupació organisme         70000 non-null  object 
 3   Organisme contractant       70000 non-null  object 
 4   Procediment d’adjudicació   70000 non-null  object 
 5   Tipus de contracte          70000 non-null  object 
 6   Número de lot               70000 non-null  int64  
 7   Adjudicatari                70000 non-null  object 
 8   Import d’adjudicació        70000 non-null  float64
 9   Durada dies                 70000 non-null  int64  
 10  Durada mesos                70000 non-null  int64  
 11  Durada anys                 70000 non-null  int64  
 12  Tipus de liquidació         70000 non-null  object 
 13  Import de la liquidació     70

In [107]:
columnas_no_numericas = df_reg.select_dtypes(exclude=['number']).columns

for col in columnas_no_numericas:
    print(f'\n {col}: {df_reg[col].nunique()}')


 Àmbit organitzatiu: 3

 Agrupació organisme: 642

 Organisme contractant: 952

 Procediment d’adjudicació: 9

 Tipus de contracte: 7

 Adjudicatari: 27305

 Tipus de liquidació: 2


In [ ]:
#Para aplicar la escala logaritmica, el codigo es el siguiente
df_nulos[target] = np.log1p(df_nulos[target])
plt.figure(figsize=(10, 6))
sns.kdeplot(data=df_nulos, x=target, fill=True)
plt.title('Distribución de la Target (KDE) - Log transformada')
plt.show()
# Y esto para ver la distribucion de la target
target = "Import de la liquidació"